# Numerical Solution of ODEs: The Leapfrog Particle Pusher for an IVP #

In [9]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

omega_p = 1.0
x0, v0, T = 0.5, 0.0, 20.0

def x_exact(t): return 0.5 * np.cos(omega_p * t)
def v_exact(t): return -0.5 * np.sin(omega_p * t)

# The leapfrog step function #

In [10]:
def accel(x, omega_p=omega_p):
    return -omega_p**2 * x

def leapfrog(x, v, h, omega_p=omega_p):
    v_new = v + h * accel(x, omega_p)   
    x_new = x + h * v_new              
    return x_new, v_new

In [11]:
h = 0.1
xp = np.array([0.5, 0.3, -0.2, 0.05])        # 4 particles
vp = np.zeros_like(xp) - 0.5*h*accel(xp)      # staggered start (Task 2)
N = int(round(T / h))
for _ in range(N):
    xp, vp = leapfrog(xp, vp, h)

print("Particle positions at t = 20 (leapfrog):", np.round(xp, 5))
print("Exact                              :", np.round(np.array([0.5, 0.3, -0.2, 0.05])*np.cos(T), 5))

Particle positions at t = 20 (leapfrog): [ 0.20023  0.12014 -0.08009  0.02002]
Exact                              : [ 0.20404  0.12242 -0.08162  0.0204 ]


## Initializing the half step

The initial condition gives $v$ at $t=0$, but leapfrog needs $v^{-1/2}$. Move the velocity back half a step:

$$
v^{-1/2} = v^0 - \frac{h}{2}\,a(x^0), \qquad a(x) = -\omega_p^2 x.
$$

This is one Euler step of size $h/2$. Its local error is $O(h^2)$ and it is applied only once, so it does not reduce the global second-order accuracy.

In [12]:
def init_half_step(x0, v0, h, omega_p=omega_p):

    return v0 - 0.5 * h * accel(x0, omega_p)

def integrate(h, T=T, x0=x0, v0=v0, staggered=True, omega_p=omega_p, nsteps=None):
    N = nsteps if nsteps is not None else int(round(T / h))
    x = np.empty(N + 1)
    v = np.empty(N)
    x[0] = x0
    vh = init_half_step(x0, v0, h, omega_p) if staggered else v0
    for n in range(N):
        x[n+1], vh = leapfrog(x[n], vh, h, omega_p)
        v[n] = vh
    t_x = h * np.arange(N + 1)
    t_v = h * (np.arange(N) + 0.5)
    return t_x, x, t_v, v
print("h = 0.1:  v^{-1/2} =", init_half_step(x0, v0, 0.1),
      "   exact v(-h/2) =", v_exact(-0.05))

h = 0.1:  v^{-1/2} = 0.025    exact v(-h/2) = 0.024989584635339165
